# 👁️ RetinaGuard AI — Diabetic Retinopathy Severity Assessment

**Pipeline:**
```
Retinal Image → Quality Check → DR Prediction → Uncertainty → Grad-CAM → Referral Decision
```

**Run order:**
- Cells 1–5: Setup (always run these first after any restart)
- Cells 6–8: Dataset download (run once only)
- Cells 9–11: Image quality module
- Cells 12–13: Model training (skip if .pth already saved)
- Cells 14–17: Prediction + Uncertainty + Grad-CAM
- Cell 18: Full pipeline test
- Cell 19: Evaluation
- Cell 20: Streamlit app launch

> ⚠️ **Before starting:** Runtime → Change runtime type → T4 GPU

## 🔧 SECTION 1 — Setup (Always run after restart)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Mount Google Drive and create project folders
# WHY: All models and outputs are saved to Drive so they
#      survive Colab session restarts
# ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/RetinaGuard', exist_ok=True)
os.makedirs('/content/drive/MyDrive/RetinaGuard/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/RetinaGuard/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/RetinaGuard/outputs', exist_ok=True)
print('✅ Drive mounted and folders ready')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Install all dependencies
# WHY: These libraries are not pre-installed in Colab
#   timm     → EfficientNet model
#   grad-cam → Visual explanations
#   kaggle   → Dataset download
#   shap     → Feature importance
# ─────────────────────────────────────────────────────────
!pip install kaggle timm grad-cam shap pyngrok streamlit -q
print('✅ All dependencies installed')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Import all libraries
# WHY: Central imports so all cells below work correctly
# ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# Use GPU if available (T4 runtime needed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# DR grade labels used throughout the pipeline
DR_LABELS = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']

# Path where trained model weights are saved
MODEL_SAVE = '/content/drive/MyDrive/RetinaGuard/models/efficientnet_retina.pth'

print('✅ Libraries loaded')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Verify GPU is active
# WHY: Training without GPU takes 10x longer.
#      If this shows False, go to:
#      Runtime → Change runtime type → T4 GPU
# ─────────────────────────────────────────────────────────
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU — enable T4 GPU in Runtime settings')

## 📦 SECTION 2 — Dataset Download (Run once only)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Upload Kaggle API credentials
# WHY: Kaggle requires authentication to download datasets
# HOW: Go to kaggle.com → Settings → API → Create New Token
#      Upload the downloaded kaggle.json file when prompted
# SKIP: If dataset already downloaded to /content/aptos2019/
# ─────────────────────────────────────────────────────────
from google.colab import files
files.upload()  # Upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle API ready')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Download APTOS 2019 dataset
# WHY: APTOS 2019 has 2930 retinal images with 5 DR grades
#      It naturally contains blurry/dark images — perfect
#      for testing our quality check module
# SIZE: ~9GB, takes 5-15 mins depending on internet speed
# SKIP: If aptos2019.zip already exists in /content/
# ─────────────────────────────────────────────────────────
!kaggle datasets download -d mariaherrerot/aptos2019 -p /content/
print('✅ Download complete')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Unzip and verify dataset
# WHY: Extract images and CSV from downloaded zip file
# RESULT: Creates /content/aptos2019/ with:
#   train_1.csv              → image IDs and DR grades
#   train_images/train_images/ → 2930 retinal PNG images
# ─────────────────────────────────────────────────────────
import zipfile

print('Extracting... please wait')
with zipfile.ZipFile('/content/aptos2019.zip', 'r') as z:
    z.extractall('/content/aptos2019/')

# Load and verify CSV
df = pd.read_csv('/content/aptos2019/train_1.csv')
IMG_DIR = '/content/aptos2019/train_images/train_images/'

print(f'\n✅ Dataset ready: {len(df)} images')
print(f'\nGrade distribution (0=healthy, 4=severe):')
print(df['diagnosis'].value_counts().sort_index())

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Load dataset (run this after any restart
#           IF dataset already downloaded)
# WHY: After Colab restarts /content/ is wiped but Drive
#      persists. Dataset stays in /content/aptos2019/
#      only within the same session.
# ─────────────────────────────────────────────────────────
df = pd.read_csv('/content/aptos2019/train_1.csv')
IMG_DIR = '/content/aptos2019/train_images/train_images/'
print(f'✅ Data loaded: {len(df)} images')
print(df['diagnosis'].value_counts().sort_index())

## 📷 SECTION 3 — Image Quality Module

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — Image Quality Check function
# WHY: Poor quality images (blurry/dark) make the model
#      unreliable. We block them BEFORE prediction.
# HOW: Uses OpenCV to measure:
#   blur_score  → Laplacian variance (higher = sharper)
#   brightness  → Mean pixel value (20–230 is acceptable)
# NOTE: We crop the center 60% to ignore the dark circular
#       border common in retinal fundus photos
# ─────────────────────────────────────────────────────────
def check_image_quality(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Crop center to avoid dark border skewing scores
    h, w = gray.shape
    cy, cx = h//2, w//2
    gray = gray[cy-h//3:cy+h//3, cx-w//3:cx+w//3]

    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = gray.mean()

    # Thresholds tuned for retinal fundus images
    passed = (blur_score >= 10 and
              brightness >= 20 and
              brightness <= 230)

    return passed, round(float(blur_score), 2), round(float(brightness), 2)

# Test on 5 images
print('Testing quality check on 5 sample images:\n')
for i in range(5):
    img_path = f'{IMG_DIR}{df.iloc[i]["id_code"]}.png'
    passed, blur, bright = check_image_quality(img_path)
    verdict = '✅ PASS' if passed else '🚫 FAIL'
    print(f'Image {i} | Grade {df.iloc[i]["diagnosis"]} | Blur: {blur} | Brightness: {bright} | {verdict}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 10 — Run quality check on all 2930 images
# WHY: To understand what % of real-world images fail
#      quality check — this becomes a key metric in report
# EXPECTED: ~28% fail (realistic for clinical datasets)
# ─────────────────────────────────────────────────────────
pass_count = 0
fail_count = 0

for i in range(len(df)):
    img_path = f'{IMG_DIR}{df.iloc[i]["id_code"]}.png'
    passed, _, _ = check_image_quality(img_path)
    if passed:
        pass_count += 1
    else:
        fail_count += 1

total = pass_count + fail_count
print(f'Total images : {total}')
print(f'✅ PASS      : {pass_count} ({pass_count/total*100:.1f}%)')
print(f'🚫 FAIL      : {fail_count} ({fail_count/total*100:.1f}%)')

## 🧠 SECTION 4 — Model Training (Skip if .pth already saved)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 11 — Build dataset class and data loaders
# WHY: PyTorch needs a Dataset class to load images in
#      batches during training
# KEY CHOICES:
#   Resize 224x224  → EfficientNet input size
#   Flip/ColorJitter → Data augmentation to prevent overfitting
#   Normalize        → ImageNet mean/std (pretrained model expects this)
#   80/20 split      → 2344 train, 586 validation
# ─────────────────────────────────────────────────────────
class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = f"{self.img_dir}{self.df.iloc[idx]['id_code']}.png"
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx]['diagnosis']
        if self.transform:
            image = self.transform(image)
        return image, label

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_df = df.sample(frac=0.8, random_state=42)
val_df   = df.drop(train_df.index)

train_ds = APTOSDataset(train_df, IMG_DIR, transform_train)
val_ds   = APTOSDataset(val_df,   IMG_DIR, transform_val)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} images | Val: {len(val_ds)} images')

# EfficientNet-B0 pretrained on ImageNet, fine-tuned for 5 DR classes
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5)
model = model.to(DEVICE)

# Weighted loss: penalizes errors on rare classes (Severe/Mild)
# more than common classes (No DR)
class_counts = df['diagnosis'].value_counts().sort_index().values
weights = torch.tensor(1.0 / class_counts, dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f'✅ Model ready on {DEVICE}')
print('✅ Weighted loss and Adam optimizer ready')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 12 — Train EfficientNet-B0
# WHY: Fine-tune pretrained model on APTOS retinal images
# TIME: ~45-60 mins for 15 epochs on T4 GPU
# SAVES: Best model checkpoint to Drive after each epoch
#        that improves validation loss
# SKIP: If efficientnet_retina.pth already exists in Drive
#       → go directly to CELL 13 to load saved model
# ─────────────────────────────────────────────────────────
EPOCHS = 15
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    # Validation phase
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    train_acc      = train_correct / train_total * 100
    val_acc        = sum(p==l for p,l in zip(all_preds,all_labels)) / len(all_labels) * 100
    f1             = f1_score(all_labels, all_preds, average='macro') * 100
    avg_train_loss = train_loss / len(train_dl)
    avg_val_loss   = val_loss   / len(val_dl)

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'Train Loss: {avg_train_loss:.4f} | '
          f'Val Loss: {avg_val_loss:.4f} | '
          f'Train Acc: {train_acc:.1f}% | '
          f'Val Acc: {val_acc:.1f}% | '
          f'Macro F1: {f1:.1f}%')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE)
        print(f'  💾 Best model saved (val loss: {best_val_loss:.4f})')

print('\n📊 Final Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))
print('\n📊 Per-class F1:')
for label, score in zip(DR_LABELS, f1_score(all_labels, all_preds, average=None)):
    print(f'  {label:15s}: {score*100:.1f}%')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 13 — Load saved model (run this instead of CELL 12
#            if training already done)
# WHY: Loads the best checkpoint saved during training
#      so you never need to retrain
# FOR FRIEND: She runs only this cell to load the model
#             Change MODEL_PATH to where she saved the .pth
# ─────────────────────────────────────────────────────────
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5)
model.load_state_dict(torch.load(MODEL_SAVE, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print('✅ Model loaded from saved checkpoint')
print(f'   Path: {MODEL_SAVE}')

## 🎯 SECTION 5 — Prediction + Uncertainty + Grad-CAM

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 14 — Image transform for inference
# WHY: Model expects 224x224 normalized tensors
#      Same normalization as training (ImageNet stats)
# ─────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
print('✅ Transform ready')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 15 — Monte Carlo Dropout for uncertainty estimation
# WHY: Standard models give overconfident predictions.
#      MC Dropout runs the model 30 times with dropout ON
#      and measures how much predictions vary (variance).
#      High variance = model is unsure = refer to doctor
# HOW: Keep dropout layers in train() mode during inference
#      Run 30 forward passes, compute mean and variance
# ─────────────────────────────────────────────────────────
def enable_dropout(model):
    # Keep dropout ON during inference for uncertainty
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

def predict_with_uncertainty(img_tensor, n_passes=30):
    enable_dropout(model)
    preds = []
    with torch.no_grad():
        for _ in range(n_passes):  # 30 stochastic forward passes
            out = torch.softmax(model(img_tensor), dim=1)
            preds.append(out.cpu().numpy())
    preds = np.array(preds)          # shape: (30, 1, 5)
    mean_pred   = preds.mean(axis=0)[0]   # average prediction
    uncertainty = preds.var(axis=0).mean() # variance = uncertainty
    return mean_pred, float(uncertainty)

UNCERTAINTY_THRESHOLD = 0.01  # above this → refer to doctor
print('✅ MC Dropout uncertainty ready')
print(f'   Threshold: {UNCERTAINTY_THRESHOLD} (predictions above this get referred)')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 16 — Grad-CAM visual explanation
# WHY: Doctors need to know WHICH part of the retina the
#      model focused on. Red/yellow areas = high attention.
#      This builds trust and helps doctors verify the decision.
# HOW: Computes gradients of the predicted class score
#      with respect to the last conv layer activations
# ─────────────────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target the last convolutional layer of EfficientNet
target_layer = [model.conv_head]

def generate_gradcam(img_path):
    img = Image.open(img_path).convert('RGB')
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized) / 255.0
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        output = torch.softmax(model(tensor), dim=1)
    grade      = output.argmax(dim=1).item()
    confidence = output[0][grade].item()

    cam = GradCAM(model=model, target_layers=target_layer)
    grayscale_cam = cam(
        input_tensor=tensor,
        targets=[ClassifierOutputTarget(grade)]
    )[0]

    visualization = show_cam_on_image(
        img_array.astype(np.float32),
        grayscale_cam,
        use_rgb=True
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img_resized)
    axes[0].set_title('Original Retinal Image')
    axes[0].axis('off')
    axes[1].imshow(visualization)
    axes[1].set_title(f'Grad-CAM | Grade {grade}: {DR_LABELS[grade]} | Conf: {confidence*100:.1f}%')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Grade: {grade} — {DR_LABELS[grade]} | Confidence: {confidence*100:.1f}%')

# Test on one sample image
sample_path = f'{IMG_DIR}{df.iloc[2]["id_code"]}.png'
generate_gradcam(sample_path)

## 🔬 SECTION 6 — Full Pipeline + Evaluation

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 17 — Full end-to-end pipeline
# WHY: Combines all 4 modules into one function:
#   1. Quality check  → block bad images first
#   2. DR prediction  → EfficientNet grade 0-4
#   3. Referral logic → uncertain/severe → doctor
#   4. Grad-CAM       → visual explanation
# USE: Call run_full_pipeline('path/to/image.png')
# ─────────────────────────────────────────────────────────
def run_full_pipeline(img_path):
    print('='*50)
    print('🔬 RetinaGuard AI — Full Pipeline')
    print('='*50)

    # Step 1: Block poor quality images before any AI prediction
    passed, blur, bright = check_image_quality(img_path)
    print(f'\n📷 STEP 1 — Image Quality Check')
    print(f'   Blur Score  : {blur}  (min=10 required)')
    print(f'   Brightness  : {bright}  (range 20-230 required)')

    if not passed:
        print(f'   Verdict     : 🚫 FAIL')
        print(f'\n🚨 REFER — Poor image quality')
        print(f'   → Please recapture or send for human review')
        print('='*50)
        return
    print(f'   Verdict     : ✅ PASS')

    # Step 2: Predict DR grade with uncertainty
    print(f'\n🧠 STEP 2 — DR Severity Prediction (30 MC passes)')
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    probs, uncertainty = predict_with_uncertainty(tensor)
    grade      = np.argmax(probs)
    confidence = probs[grade]
    print(f'   DR Grade    : {grade} — {DR_LABELS[grade]}')
    print(f'   Confidence  : {confidence*100:.1f}%')
    print(f'   Uncertainty : {uncertainty:.5f}  (threshold={UNCERTAINTY_THRESHOLD})')

    # Step 3: Referral decision based on uncertainty and grade
    print(f'\n🏥 STEP 3 — Referral Decision')
    if uncertainty > UNCERTAINTY_THRESHOLD:
        print(f'   🚨 REFER — High uncertainty → Human review needed')
    elif grade >= 2:
        print(f'   🚨 REFER — Grade {grade} detected → Ophthalmologist review needed')
    else:
        print(f'   🟢 AUTO  — Low severity, prediction reliable')

    # Step 4: Grad-CAM visual explanation for doctor
    print(f'\n🗺️  STEP 4 — Grad-CAM Explanation (red=high attention)')
    img_resized = img.resize((224, 224))
    img_array   = np.array(img_resized) / 255.0
    cam = GradCAM(model=model, target_layers=target_layer)
    grayscale_cam = cam(
        input_tensor=tensor,
        targets=[ClassifierOutputTarget(grade)]
    )[0]
    visualization = show_cam_on_image(
        img_array.astype(np.float32), grayscale_cam, use_rgb=True
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img_resized)
    axes[0].set_title('Original Retinal Image')
    axes[0].axis('off')
    axes[1].imshow(visualization)
    axes[1].set_title(f'Grad-CAM | Grade {grade} — {DR_LABELS[grade]}')
    axes[1].axis('off')
    plt.suptitle(
        f'RetinaGuard AI | Conf: {confidence*100:.1f}% | Uncertainty: {uncertainty:.5f}',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
    print('='*50)

# Test on 3 images covering different grades
for i in [0, 5, 10]:
    sample_path = f'{IMG_DIR}{df.iloc[i]["id_code"]}.png'
    print(f'\nImage {i} — True Grade: {df.iloc[i]["diagnosis"]}')
    run_full_pipeline(sample_path)
    print()

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 18 — Evaluate on full validation set (586 images)
# WHY: Measures real-world performance of the pipeline
# METRICS:
#   Quality failed  → % images blocked before prediction
#   Referred        → % flagged for human review
#   Accuracy        → % correct on auto-predicted images
#   Macro F1        → balanced score across all 5 grades
#   Confusion matrix → shows which grades get confused
# ─────────────────────────────────────────────────────────
model.eval()
all_preds      = []
all_labels     = []
referred       = 0
quality_failed = 0

val_df = df.drop(df.sample(frac=0.8, random_state=42).index)
print(f'Evaluating {len(val_df)} validation images...')

for i in range(len(val_df)):
    img_path   = f'{IMG_DIR}{val_df.iloc[i]["id_code"]}.png'
    true_label = val_df.iloc[i]['diagnosis']

    passed, _, _ = check_image_quality(img_path)
    if not passed:
        quality_failed += 1
        continue

    img    = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    probs, uncertainty = predict_with_uncertainty(tensor)
    grade  = np.argmax(probs)

    if uncertainty > UNCERTAINTY_THRESHOLD:
        referred += 1
        continue

    all_preds.append(grade)
    all_labels.append(true_label)

total_val      = len(val_df)
auto_predicted = len(all_preds)

print(f'\n📊 EVALUATION RESULTS')
print(f'{"="*50}')
print(f'Total val images    : {total_val}')
print(f'Quality failed      : {quality_failed} ({quality_failed/total_val*100:.1f}%)')
print(f'Referred (uncertain): {referred} ({referred/total_val*100:.1f}%)')
print(f'Auto predicted      : {auto_predicted} ({auto_predicted/total_val*100:.1f}%)')
print(f'\n📈 On auto-predicted images:')
print(f'Accuracy  : {sum(p==l for p,l in zip(all_preds,all_labels))/len(all_labels)*100:.1f}%')
print(f'Macro F1  : {f1_score(all_labels, all_preds, average="macro")*100:.1f}%')
print(f'\n📊 Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))
print(f'\n📊 Per-class Report:')
print(classification_report(
    all_labels, all_preds,
    target_names=DR_LABELS
))

## 🚀 SECTION 7 — Streamlit App Demo

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 19 — Launch Streamlit demo app
# WHY: Interactive web UI for demo/presentation
#      Anyone can upload a retinal image and see results
# HOW: Uses ngrok to create a public URL from Colab
# STEPS:
#   1. Get free token at https://dashboard.ngrok.com
#   2. Paste your token below where it says YOUR_TOKEN
#   3. Run cell — open the printed URL in any browser
# NOTE: URL is only active while this Colab session runs
# ─────────────────────────────────────────────────────────
import subprocess, time
from pyngrok import ngrok

# Copy app files to local Colab storage
!cp /content/drive/MyDrive/RetinaGuard/app.py /content/app.py
!cp /content/drive/MyDrive/RetinaGuard/models/efficientnet_retina.pth /content/efficientnet_retina.pth

# Kill any previous instances
!pkill -f streamlit
!pkill -f ngrok
time.sleep(2)

# Set your ngrok auth token (get free at dashboard.ngrok.com)
!ngrok authtoken YOUR_TOKEN_HERE

# Start Streamlit server
subprocess.Popen(
    ['streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(8)

# Create public URL
public_url = ngrok.connect(8501)
print(f'\n✅ App is live!')
print(f'🌐 Open this URL: {public_url}')
print(f'\n📤 Share this link for demo — works in any browser')
print(f'⚠️  Link stays active only while this Colab runs')